In [1]:
import ir_datasets
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

dataset = ir_datasets.load("wikir/en1k/training")
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

nltk.download("punkt_tab")
nltk.download("wordnet")

doc_generator = (doc.text for doc in dataset.docs_iter())
lemmatized_docs = []
lematizer = WordNetLemmatizer()

for text in doc_generator:
    tokens = word_tokenize(text)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_docs.append(" ".join(lemmatized_words))

print("The texts lematized!")

docs generator created!


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tahas44/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


The texts lematized!


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf_vect = TfidfVectorizer(stop_words="english")
tf_idf_matrix = tf_idf_vect.fit_transform(lemmatized_docs)
print("TF-IDF matrix ready!")

TF-IDF matrix ready!


In [3]:
queries = [query.text for query in dataset.queries_iter()]

lemmatized_queries = []

for query in queries:
    tokens = word_tokenize(query)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_queries.append(" ".join(lemmatized_words))

query_vectors = tf_idf_vect.transform(lemmatized_queries)

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vectors, tf_idf_matrix)

In [5]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("necessary dicts ready!")

necessary dicts ready!


In [6]:
similarities

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.04209614, 0.01107245, ..., 0.        , 0.12598742,
        0.        ]], shape=(1444, 369721))

In [7]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [8]:
from helper import create_AP, create_ndcg, create_statistical_columns, print_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, similarities)
df = create_AP(df, qrels_dict, doc_dict, similarities)
df = create_ndcg(df, doc_dict, similarities, score_doc_dict)

print_columns(df)

recall_5_mean: 12.46931231563617
recall_5_std: 13.857174147785404
recall_5_max: 83.33333333333334
recall_5_min: 0.0
recall_10_mean: 17.30007008682119
recall_10_std: 18.044377219380316
recall_10_max: 100.0
recall_10_min: 0.0
precision_5_mean: 25.844875346260388
precision_5_std: 22.502317614949103
precision_5_max: 100.0
precision_5_min: 0.0
precision_10_mean: 18.912742382271468
precision_10_std: 16.748786026490652
precision_10_max: 90.0
precision_10_min: 0.0
f_score_5_mean: 15.603934341756686
f_score_5_std: 16.082301038046463
f_score_5_max: 90.9090909090909
f_score_5_min: 0.0
f_score_10_mean: 16.23472656199969
f_score_10_std: 15.395549936714758
f_score_10_max: 88.88888888888889
f_score_10_min: 0.0
MAP_5: 0.09776628078657493
MAP_10: 0.11727319721916266
NDCG_5_mean: 0.47661930500799454
NDCG_5_std: 0.3438484078665033
NDCG_5_max: 1.0000000000000002
NDCG_5_min: 0.0
NDCG_10_mean: 0.4760872201170601
NDCG_10_std: 0.33653236896747757
NDCG_10_max: 1.0000000000000002
NDCG_10_min: 0.0
